# Finance Project — Exploratory Data Analysis
**Phase 2 EDA**: Validate that signals are meaningful before feeding them to models.

Sections:
1. Setup & data loading
2. OHLCV sanity checks
3. Technical indicator distributions
4. Sector correlation heatmaps
5. Rolling volatility analysis
6. Drawdown analysis
7. Volume anomaly detection
8. Fundamental features overview
9. Signal quality check (IC / predictive power preview)

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))   # project root

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import text

from config.settings import SECTORS, ALL_TICKERS
from data.storage.db_client import get_conn
from features.analytics import (
    compute_correlation_matrix,
    compute_drawdown_series,
    compute_volume_anomalies,
)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)

# ── convenience helpers ──────────────────────────────────────────────────────
def sql(query, **params):
    with get_conn() as conn:
        return pd.read_sql(text(query), conn, params=params or None)

SECTOR_COLORS = {
    'Technology': '#4C72B0', 'Healthcare': '#DD8452',
    'Financials': '#55A868', 'Consumer': '#C44E52', 'Energy': '#8172B3',
}

print('Setup complete')

---
## 1 · OHLCV Sanity Checks

In [ ]:
# Row counts per ticker
counts = sql("""
    SELECT ticker, COUNT(*) AS rows,
           MIN(date) AS earliest, MAX(date) AS latest
    FROM ohlcv GROUP BY ticker ORDER BY rows DESC
""")
print(f"Tickers with data: {len(counts)}")
print(f"Median row count : {counts['rows'].median():.0f}")
print(f"Universe coverage: {len(counts)}/{len(ALL_TICKERS)} tickers")
counts.head(10)

In [ ]:
# Check for missing dates (gaps in OHLCV series)
gaps = sql("""
    SELECT ticker,
           date - LAG(date) OVER (PARTITION BY ticker ORDER BY date) AS gap_days
    FROM ohlcv
""")
large_gaps = gaps[gaps['gap_days'] > 5].dropna()
print(f"Date gaps > 5 trading days: {len(large_gaps)}")
large_gaps.sort_values('gap_days', ascending=False).head(10)

In [ ]:
# Price history of sample tickers
sample = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'XOM']
prices = sql("""
    SELECT ticker, date, adj_close FROM ohlcv
    WHERE ticker = ANY(:tickers) ORDER BY ticker, date
""", tickers=sample)
prices['date'] = pd.to_datetime(prices['date'])

fig, ax = plt.subplots(figsize=(14, 5))
for tkr, grp in prices.groupby('ticker'):
    # Normalise to 100 at start
    norm = grp.set_index('date')['adj_close']
    (norm / norm.iloc[0] * 100).plot(ax=ax, label=tkr)
ax.set_title('Normalised Adjusted Close (base=100)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2 · Technical Indicator Distributions

In [ ]:
tf = sql("""
    SELECT t.ticker, t.date, t.rsi_14, t.macd, t.macd_hist,
           t.bb_pct, t.atr_14, t.volume_zscore,
           t.daily_return, t.rolling_vol_20, s.sector
    FROM technical_features t
    JOIN stocks s USING (ticker)
    WHERE t.date >= CURRENT_DATE - INTERVAL '2 years'
""")
tf['date'] = pd.to_datetime(tf['date'])
print(f"Technical feature rows: {len(tf):,}")
tf.describe().round(4)

In [ ]:
# RSI distribution across sectors
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
tf['rsi_14'].dropna().hist(bins=60, ax=axes[0], color='steelblue', edgecolor='none')
axes[0].axvline(30, color='red', linestyle='--', label='Oversold (30)')
axes[0].axvline(70, color='green', linestyle='--', label='Overbought (70)')
axes[0].set_title('RSI(14) Distribution — All Tickers')
axes[0].legend()

for sector, grp in tf.groupby('sector'):
    grp['rsi_14'].dropna().plot.kde(ax=axes[1], label=sector, color=SECTOR_COLORS.get(sector))
axes[1].set_xlim(0, 100)
axes[1].set_title('RSI(14) KDE by Sector')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Bollinger Band %B — where is price within the bands?
fig, ax = plt.subplots(figsize=(14, 4))
tf['bb_pct'].dropna().clip(-0.5, 1.5).hist(bins=80, ax=ax, color='darkorange', edgecolor='none')
ax.axvline(0, color='red', linestyle='--', label='Lower band')
ax.axvline(1, color='green', linestyle='--', label='Upper band')
ax.set_title('Bollinger Band %B Distribution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# MACD histogram — positive = bullish momentum, negative = bearish
macd_latest = (
    tf.sort_values('date')
      .groupby('ticker')
      .last()
      .reset_index()[['ticker', 'macd_hist', 'sector']]
      .dropna()
      .sort_values('macd_hist')
)

fig = px.bar(
    macd_latest,
    x='ticker', y='macd_hist', color='sector',
    color_discrete_map=SECTOR_COLORS,
    title='Latest MACD Histogram by Ticker',
    height=400,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## 3 · Sector Correlation Heatmaps

In [ ]:
START = '2022-01-01'

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, (sector, tickers) in enumerate(SECTORS.items()):
    corr = compute_correlation_matrix(tickers, start=START)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr, ax=axes[i], mask=mask,
        cmap='RdYlGn', center=0, vmin=-1, vmax=1,
        annot=len(tickers) <= 10, fmt='.2f', annot_kws={'size': 6},
        linewidths=0.3, square=True, cbar_kws={'shrink': 0.6},
    )
    axes[i].set_title(f'{sector} — Log-Return Correlations\n(since {START})', fontsize=10)
    axes[i].tick_params(axis='both', labelsize=7)

# Full universe in last panel
corr_all = compute_correlation_matrix(ALL_TICKERS, start=START)
sns.heatmap(
    corr_all, ax=axes[5], cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    xticklabels=False, yticklabels=False, cbar_kws={'shrink': 0.6},
)
axes[5].set_title('Full Universe Correlation Matrix', fontsize=10)

plt.suptitle('Sector Return Correlations', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-sector average correlation over time (rolling 63-day window)
prices_wide = sql("""
    SELECT date, ticker, log_return
    FROM technical_features
    WHERE date >= :start ORDER BY date
""", start=START)
prices_wide['date'] = pd.to_datetime(prices_wide['date'])
pivot = prices_wide.pivot_table(index='date', columns='ticker', values='log_return')

# Average pairwise correlation in each sector
fig, ax = plt.subplots(figsize=(14, 4))
for sector, tickers in SECTORS.items():
    sector_cols = [t for t in tickers if t in pivot.columns]
    if len(sector_cols) < 2:
        continue
    rolling_corr = (
        pivot[sector_cols]
        .rolling(63)
        .corr()
        .groupby(level=0)
        .apply(lambda m: m.values[np.triu_indices_from(m.values, k=1)].mean())
    )
    rolling_corr.plot(ax=ax, label=sector, color=SECTOR_COLORS.get(sector))

ax.set_title('Rolling 63-day Average Intra-Sector Correlation')
ax.set_ylabel('Avg Pairwise Correlation')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4 · Rolling Volatility Analysis

In [ ]:
# Average 20-day realised vol per sector over time
vol_data = sql("""
    SELECT t.date, s.sector, AVG(t.rolling_vol_20) AS avg_vol
    FROM technical_features t
    JOIN stocks s USING (ticker)
    WHERE t.date >= :start AND t.rolling_vol_20 IS NOT NULL
    GROUP BY t.date, s.sector
    ORDER BY t.date
""", start=START)
vol_data['date'] = pd.to_datetime(vol_data['date'])

fig, ax = plt.subplots(figsize=(14, 5))
for sector, grp in vol_data.groupby('sector'):
    grp.set_index('date')['avg_vol'].plot(
        ax=ax, label=sector, color=SECTOR_COLORS.get(sector),
    )
ax.set_title('Average 20-Day Realised Volatility by Sector (annualised)')
ax.set_ylabel('Volatility')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Vol distribution — cross-sectional box plot per sector (latest snapshot)
latest_vol = sql("""
    SELECT DISTINCT ON (t.ticker)
           t.ticker, t.rolling_vol_20, s.sector
    FROM technical_features t
    JOIN stocks s USING (ticker)
    WHERE t.rolling_vol_20 IS NOT NULL
    ORDER BY t.ticker, t.date DESC
""")

fig = px.box(
    latest_vol.dropna(), x='sector', y='rolling_vol_20',
    color='sector', color_discrete_map=SECTOR_COLORS,
    points='all', hover_name='ticker',
    title='Latest 20-Day Realised Vol Distribution by Sector',
    labels={'rolling_vol_20': 'Annualised Vol'},
)
fig.show()

---
## 5 · Drawdown Analysis

In [ ]:
# Drawdown for a few representative names
sample_dd = ['AAPL', 'NVDA', 'JPM', 'XOM', 'UNH']

fig, axes = plt.subplots(len(sample_dd), 1, figsize=(14, 3 * len(sample_dd)), sharex=True)
for ax, ticker in zip(axes, sample_dd):
    dd = compute_drawdown_series(ticker, start=START)
    dd.plot(ax=ax, color='crimson', linewidth=0.8)
    ax.fill_between(dd.index, dd.values, 0, color='crimson', alpha=0.25)
    ax.set_ylabel(ticker, fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.axhline(0, color='black', linewidth=0.5)
    max_dd = dd.min()
    ax.set_title(f'{ticker} — Max Drawdown: {max_dd:.1%}', fontsize=9)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.suptitle('Drawdown Series (from peak)', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Max drawdown ranking — worst performers over the window
dd_summary = []
for ticker in ALL_TICKERS:
    dd = compute_drawdown_series(ticker, start=START)
    if not dd.empty:
        sector = next((s for s, tks in SECTORS.items() if ticker in tks), 'Unknown')
        dd_summary.append({'ticker': ticker, 'sector': sector, 'max_drawdown': dd.min()})

dd_df = pd.DataFrame(dd_summary).sort_values('max_drawdown')

fig = px.bar(
    dd_df.head(30), x='ticker', y='max_drawdown',
    color='sector', color_discrete_map=SECTOR_COLORS,
    title=f'Top 30 Worst Max Drawdowns (since {START})',
    labels={'max_drawdown': 'Max Drawdown'},
    height=400,
)
fig.update_layout(yaxis_tickformat='.0%', xaxis_tickangle=-45)
fig.show()

---
## 6 · Volume Anomaly Detection

In [ ]:
# Fetch all volume spikes (|z-score| > 2) in the last year
anomalies = compute_volume_anomalies(ALL_TICKERS, start='2023-01-01', zscore_threshold=2.0)
print(f"Total volume anomalies: {len(anomalies):,}")

fig = px.scatter(
    anomalies,
    x='date', y='volume_zscore',
    color='sector', hover_name='ticker',
    color_discrete_map=SECTOR_COLORS,
    title='Volume Z-Score Anomalies (|z| > 2) — Last 12 Months',
    opacity=0.6, height=450,
)
fig.add_hline(y=2, line_dash='dash', line_color='red', annotation_text='+2σ')
fig.add_hline(y=-2, line_dash='dash', line_color='red', annotation_text='-2σ')
fig.show()

In [ ]:
# Which tickers have the most volume anomaly days?
spike_counts = (
    anomalies.groupby(['ticker', 'sector'])
    .size()
    .reset_index(name='spike_days')
    .sort_values('spike_days', ascending=False)
    .head(20)
)

fig = px.bar(
    spike_counts, x='ticker', y='spike_days',
    color='sector', color_discrete_map=SECTOR_COLORS,
    title='Tickers with Most Volume Spike Days',
    height=400,
)
fig.show()

---
## 7 · Fundamental Features Overview

In [ ]:
fund = sql("""
    SELECT f.*, s.sector
    FROM fundamental_features f
    JOIN stocks s USING (ticker)
""")
print(f"Fundamental rows: {len(fund):,}")
fund.describe().round(2)

In [ ]:
# Latest P/E ratios by sector
latest_fund = (
    fund.sort_values('period')
        .groupby('ticker')
        .last()
        .reset_index()
)

fig = px.box(
    latest_fund.dropna(subset=['pe_ratio']).query('pe_ratio > 0 and pe_ratio < 100'),
    x='sector', y='pe_ratio', color='sector',
    color_discrete_map=SECTOR_COLORS,
    points='all', hover_name='ticker',
    title='Trailing P/E Distribution by Sector',
)
fig.show()

In [ ]:
# Revenue QoQ growth distribution
fig, ax = plt.subplots(figsize=(14, 4))
for sector, grp in latest_fund.groupby('sector'):
    data = grp['revenue_qoq'].dropna().clip(-50, 50)
    if len(data) > 3:
        data.plot.kde(ax=ax, label=sector, color=SECTOR_COLORS.get(sector))
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Revenue QoQ Growth Distribution by Sector (%)')
ax.set_xlabel('QoQ Revenue Growth (%)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gross margin vs operating margin scatter
fig = px.scatter(
    latest_fund.dropna(subset=['gross_margin', 'operating_margin']),
    x='gross_margin', y='operating_margin',
    color='sector', text='ticker',
    color_discrete_map=SECTOR_COLORS,
    title='Gross Margin vs Operating Margin (latest quarter)',
    labels={'gross_margin': 'Gross Margin (%)', 'operating_margin': 'Operating Margin (%)'},
    hover_name='ticker', height=500,
)
fig.update_traces(textposition='top center', textfont_size=8)
fig.show()

---
## 8 · Signal Quality Check (IC Preview)
> **Information Coefficient (IC)**: Spearman rank correlation between a feature value on day T and the 1-day forward return on day T+1.
> IC > 0.05 is considered useful in practice. This is a quick sanity check before we build models.

In [ ]:
from scipy.stats import spearmanr

# Load features + 1-day forward return
signal_df = sql("""
    SELECT t.ticker, t.date,
           t.rsi_14, t.macd_hist, t.bb_pct, t.volume_zscore,
           t.rolling_vol_20,
           LEAD(t.daily_return) OVER (PARTITION BY t.ticker ORDER BY t.date) AS fwd_return_1d
    FROM technical_features t
    WHERE t.date >= :start
""", start=START)
signal_df = signal_df.dropna(subset=['fwd_return_1d'])

features = ['rsi_14', 'macd_hist', 'bb_pct', 'volume_zscore', 'rolling_vol_20']
ic_results = []

# Compute IC per feature, per day (then average)
signal_df['date'] = pd.to_datetime(signal_df['date'])
for feat in features:
    sub = signal_df[['date', feat, 'fwd_return_1d']].dropna()
    daily_ic = []
    for dt, grp in sub.groupby('date'):
        if len(grp) < 10:
            continue
        ic, _ = spearmanr(grp[feat], grp['fwd_return_1d'])
        daily_ic.append(ic)
    ic_series = pd.Series(daily_ic)
    ic_results.append({
        'feature': feat,
        'mean_ic':    ic_series.mean(),
        'ic_std':     ic_series.std(),
        'icir':       ic_series.mean() / ic_series.std() if ic_series.std() > 0 else 0,
        'pct_positive': (ic_series > 0).mean() * 100,
    })

ic_df = pd.DataFrame(ic_results).sort_values('icir', ascending=False)
print('\nIC Summary (1-day forward return):')
ic_df.round(4)

In [ ]:
fig = px.bar(
    ic_df, x='feature', y='mean_ic',
    error_y='ic_std',
    color='mean_ic', color_continuous_scale='RdYlGn',
    title='Mean Information Coefficient (1-day fwd return)',
    labels={'mean_ic': 'Mean IC'},
    height=400,
)
fig.add_hline(y=0.05, line_dash='dash', annotation_text='IC=0.05 threshold')
fig.add_hline(y=-0.05, line_dash='dash')
fig.show()

---
## Summary

| Check | Expected outcome |
|---|---|
| OHLCV coverage | ≥ 95% of tickers with 500+ rows |
| RSI distribution | Roughly uniform 30–70, spikes at extremes |
| Intra-sector correlation | Tech > 0.6, Energy > 0.5 on average |
| Rolling vol | Energy highest, Consumer lowest |
| Volume anomalies | Concentrated around earnings dates |
| IC (RSI, MACD hist) | Should show weak but non-zero signal |

If any of the above look wrong, inspect the pipeline logs and re-run the relevant stage.